# Topic-Matched Harm Content Analysis

Uses the matched harmful/benign pair structure to ask:
**does the model encode harm content beyond topic?**

Harmful and benign conversations share the same `pair_id` (same JBB topic),
same attack structure, same turn position. The only difference is whether
the objective is harmful or benign.

## Two analyses

**1. Concordance** — train a reading vector on `harmful_rep - benign_rep`
diff vectors, evaluate on held-out topics (GroupKFold by pair_id).
Concordance = % of held-out pairs where the vector scores harmful > benign.
If this is well above 0.5, the vector generalises across topics.

**2. PCA** — project all harmful and benign turns into 2D.
Do harmful and benign form separate clusters even for the same topic?
Draw lines connecting matched pairs to make same-topic structure visible.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sns.set_theme(style="whitegrid", font_scale=1.1)
FIG_DIR = repo_root / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

REPR_ROOT     = repo_root / "data" / "representations"
LAYER_INDICES = [1, 16, 32]
LAYER_NAMES   = [f"Layer {l}" for l in LAYER_INDICES]
LAYER_COLORS  = ["#4e79a7", "#f28e2b", "#e15759"]

print("Imports OK")

## Load and match data

In [ ]:
def load_repr(folder):
    d = REPR_ROOT / folder
    states = np.load(str(d / "hidden_states.npy"), mmap_mode="r").astype(np.float32)
    meta   = pd.read_parquet(d / "metadata.parquet").reset_index(drop=True)
    return states, meta

def get_layer(states, layer_pos):
    return np.asarray(states[:, layer_pos, :], dtype=np.float32)

states_harm, meta_harm = load_repr("crescendo_harmful_10_turns_clean_faithful")
states_ben,  meta_ben  = load_repr("crescendo_benign_10_turns_clean_faithful")

meta_harm["row_harm"] = meta_harm.index
meta_ben["row_ben"]   = meta_ben.index

# Match on (pair_id, attempt, turn_idx) — same topic, same position
matched = meta_harm[["pair_id", "attempt", "turn_idx",
                      "n_context_turns", "final_verdict", "row_harm"]].merge(
    meta_ben[["pair_id", "attempt", "turn_idx", "row_ben"]],
    on=["pair_id", "attempt", "turn_idx"],
    how="inner",
).reset_index(drop=True)

print(f"Harmful rows:  {len(meta_harm)}")
print(f"Benign rows:   {len(meta_ben)}")
print(f"Matched pairs: {len(matched)}")
print(f"Unique topics: {matched['pair_id'].nunique()}")
print(f"n_context_turns distribution:")
print(matched['n_context_turns'].value_counts().sort_index())

## 1. Concordance analysis

Train reading vector on `harmful_rep - benign_rep` diff vectors.
Evaluate with 5-fold GroupKFold splitting by `pair_id` so that
no topic appears in both train and test.

**Metrics:**
- Concordance: % of held-out pairs where harmful scored > benign
- Compare reading vector (PCA) vs LR on diff vectors

In [ ]:
groups = matched["pair_id"].values
gkf    = GroupKFold(n_splits=5)
N      = len(matched)

rv_concordance = {li: [] for li in range(3)}
lr_concordance = {li: [] for li in range(3)}
rv_auc         = {li: [] for li in range(3)}

for tr, te in gkf.split(np.arange(N), groups=groups):
    for li in range(len(LAYER_INDICES)):
        h_all = get_layer(states_harm, li)
        b_all = get_layer(states_ben,  li)

        h_tr = h_all[matched["row_harm"].values[tr]]
        b_tr = b_all[matched["row_ben"].values[tr]]
        h_te = h_all[matched["row_harm"].values[te]]
        b_te = b_all[matched["row_ben"].values[te]]

        D_tr = h_tr - b_tr
        D_te = h_te - b_te

        # --- Reading vector (PCA on train diffs) ---
        sc = StandardScaler()
        D_tr_sc = sc.fit_transform(D_tr)
        pca = PCA(n_components=1, random_state=42)
        pca.fit(D_tr_sc)
        vec = pca.components_[0]

        h_proj = sc.transform(h_te) @ vec
        b_proj = sc.transform(b_te) @ vec
        if h_proj.mean() < b_proj.mean():
            vec = -vec
            h_proj, b_proj = -h_proj, -b_proj

        rv_concordance[li].append((h_proj > b_proj).mean())

        # AUC: can reading vector distinguish harmful from benign on test set?
        all_proj = np.concatenate([h_proj, b_proj])
        all_y    = np.concatenate([np.ones(len(h_proj)), np.zeros(len(b_proj))])
        rv_auc[li].append(roc_auc_score(all_y, all_proj))

        # --- LR on diff vectors (augmented with negations) ---
        D_aug  = np.concatenate([D_tr, -D_tr])
        y_aug  = np.concatenate([np.ones(len(D_tr)), np.zeros(len(D_tr))])
        sc_lr  = StandardScaler()
        clf    = LogisticRegression(max_iter=1000, C=0.1)
        clf.fit(sc_lr.fit_transform(D_aug), y_aug)
        lr_scores = clf.predict_proba(sc_lr.transform(D_te))[:, 1]
        lr_concordance[li].append((lr_scores > 0.5).mean())

print("Cross-validated results (5-fold GroupKFold by pair_id — no topic leakage):")
print(f"\n{'Metric / Method':<35}" + "".join(f"{ln:>12}" for ln in LAYER_NAMES))
print("-" * (35 + 36))
print(f"{'Concordance — reading vector':<35}" +
      "".join(f"{np.mean(rv_concordance[li]):>12.3f}" for li in range(3)))
print(f"{'Concordance — LR on diffs':<35}" +
      "".join(f"{np.mean(lr_concordance[li]):>12.3f}" for li in range(3)))
print(f"{'AUC — reading vector':<35}" +
      "".join(f"{np.mean(rv_auc[li]):>12.3f}" for li in range(3)))
print()
print("Concordance = % of matched pairs where harmful scored > benign.")
print("0.5 = chance. Both train and test topics are disjoint.")

### Concordance by n_context_turns

Does the reading vector get better at distinguishing harmful from benign
as context depth increases? Would suggest compliance priming sharpens the signal.

In [ ]:
# Train reading vector on all data, evaluate concordance by n_context_turns
# (in-sample, just to see the pattern — not cross-validated)
li = 2  # Layer 32
h_all = get_layer(states_harm, li)
b_all = get_layer(states_ben,  li)
h_mat = h_all[matched["row_harm"].values]
b_mat = b_all[matched["row_ben"].values]
D_all = h_mat - b_mat

sc_all = StandardScaler()
pca_all = PCA(n_components=1, random_state=42)
pca_all.fit(sc_all.fit_transform(D_all))
vec_all = pca_all.components_[0]

h_scores = sc_all.transform(h_mat) @ vec_all
b_scores = sc_all.transform(b_mat) @ vec_all
if h_scores.mean() < b_scores.mean():
    vec_all = -vec_all
    h_scores, b_scores = -h_scores, -b_scores

matched["h_score"] = h_scores
matched["b_score"] = b_scores
matched["correct"] = (h_scores > b_scores).astype(int)
matched["margin"]  = h_scores - b_scores

by_k = matched.groupby("n_context_turns").agg(
    concordance=("correct", "mean"),
    mean_margin=("margin",  "mean"),
    count=("correct", "count"),
).reset_index()
by_k = by_k[by_k["count"] >= 10]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(by_k["n_context_turns"], by_k["concordance"],
       color="#e15759", alpha=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("n_context_turns")
ax.set_ylabel("Concordance")
ax.set_title("Harmful vs benign concordance\nby compliance depth (Layer 32)")
ax.set_ylim(0, 1)
for _, row in by_k.iterrows():
    ax.text(row["n_context_turns"], row["concordance"] + 0.02,
            f"n={int(row['count'])}", ha="center", fontsize=8)

ax = axes[1]
ax.bar(by_k["n_context_turns"], by_k["mean_margin"],
       color="#4e79a7", alpha=0.8)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("n_context_turns")
ax.set_ylabel("Mean score margin (harmful - benign)")
ax.set_title("Score margin by compliance depth\n(positive = harmful scored higher)")

plt.tight_layout()
plt.savefig(FIG_DIR / "topic_matched_concordance_by_k.png", dpi=150)
plt.show()

print(by_k[["n_context_turns", "concordance", "mean_margin", "count"]].to_string(index=False))

## 2. PCA visualization

Project all harmful and benign turns into 2D (Layer 32).
Do harmful and benign form distinct clusters even for matched topics?

- **Left**: all points colored by label (harmful/benign)
- **Right**: zoom into a random sample of matched pairs,
  lines connecting harmful and benign from the same (pair_id, attempt, turn_idx)

In [ ]:
li = 2  # Layer 32

# Stack all harmful + benign
X_harm = get_layer(states_harm, li)
X_ben  = get_layer(states_ben,  li)
X_all  = np.concatenate([X_harm, X_ben], axis=0)
y_all  = np.concatenate([np.ones(len(X_harm)), np.zeros(len(X_ben))])

sc_pca  = StandardScaler()
X_sc    = sc_pca.fit_transform(X_all)
pca2    = PCA(n_components=2, random_state=42)
coords  = pca2.fit_transform(X_sc)

df_all = pd.DataFrame({
    "pc1":   coords[:, 0],
    "pc2":   coords[:, 1],
    "label": y_all,
})
df_harm_pca = df_all[df_all["label"] == 1].reset_index(drop=True)
df_ben_pca  = df_all[df_all["label"] == 0].reset_index(drop=True)

print(f"PCA explained variance: PC1={pca2.explained_variance_ratio_[0]:.1%}  "
      f"PC2={pca2.explained_variance_ratio_[1]:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: all points colored by label
ax = axes[0]
ax.scatter(df_ben_pca["pc1"],  df_ben_pca["pc2"],
           color="#4e79a7", alpha=0.15, s=4, label="benign",  rasterized=True)
ax.scatter(df_harm_pca["pc1"], df_harm_pca["pc2"],
           color="#e15759", alpha=0.15, s=4, label="harmful", rasterized=True)
ax.legend(markerscale=3)
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
ax.set_title("Harmful vs benign in PCA space\n(Layer 32, all turns)")

# Right: matched pairs — lines connecting same-topic harmful/benign
ax = axes[1]
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(matched), size=min(300, len(matched)), replace=False)
sample = matched.iloc[sample_idx]

# Get PCA coords for matched rows
h_coords = pca2.transform(sc_pca.transform(X_harm[sample["row_harm"].values]))
b_coords = pca2.transform(sc_pca.transform(X_ben[sample["row_ben"].values]))

# Draw connecting lines
for h, b in zip(h_coords, b_coords):
    ax.plot([h[0], b[0]], [h[1], b[1]], color="gray", alpha=0.2, linewidth=0.5)

ax.scatter(b_coords[:, 0], b_coords[:, 1],
           color="#4e79a7", alpha=0.6, s=15, label="benign",  zorder=3)
ax.scatter(h_coords[:, 0], h_coords[:, 1],
           color="#e15759", alpha=0.6, s=15, label="harmful", zorder=3)
ax.legend(markerscale=1.5)
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
ax.set_title("Matched pairs (lines = same topic)\n(300 random pairs)")

plt.tight_layout()
plt.savefig(FIG_DIR / "topic_matched_pca.png", dpi=150)
plt.show()

### PCA colored by topic

Do harmful and benign from the same topic cluster near each other
(topic dominates) or does the harmful/benign split dominate?
Color by pair_id, use marker shape for harmful/benign.

In [ ]:
# Sample a few topics to make the plot readable
N_TOPICS = 12
rng = np.random.default_rng(0)
topic_sample = rng.choice(matched["pair_id"].unique(), size=N_TOPICS, replace=False)

fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette("tab20", N_TOPICS)

for i, pid in enumerate(topic_sample):
    rows = matched[matched["pair_id"] == pid]

    h_c = pca2.transform(sc_pca.transform(X_harm[rows["row_harm"].values]))
    b_c = pca2.transform(sc_pca.transform(X_ben[rows["row_ben"].values]))

    ax.scatter(h_c[:, 0], h_c[:, 1], color=palette[i],
               marker="o", s=30, alpha=0.7,
               label=f"pair {pid} harmful" if i == 0 else "_")
    ax.scatter(b_c[:, 0], b_c[:, 1], color=palette[i],
               marker="x", s=30, alpha=0.7,
               label=f"pair {pid} benign" if i == 0 else "_")

# Legend for markers only
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='gray', linestyle='None', label='harmful'),
    Line2D([0], [0], marker='x', color='gray', linestyle='None', label='benign'),
]
ax.legend(handles=legend_elements)
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"PCA colored by topic ({N_TOPICS} random topics)\n"
             "Same color = same topic. Circle = harmful, X = benign.")
plt.tight_layout()
plt.savefig(FIG_DIR / "topic_matched_pca_by_topic.png", dpi=150)
plt.show()
print("If same-color points cluster together: topic dominates.")
print("If circles and Xs separate regardless of color: harm content dominates.")

## Summary

In [ ]:
print("=" * 60)
print("TOPIC-MATCHED HARM CONTENT — SUMMARY")
print("=" * 60)
print(f"\nMatched pairs: {len(matched)}  |  Unique topics: {matched['pair_id'].nunique()}")
print()
print("Cross-validated concordance (GroupKFold by pair_id):")
for li in range(3):
    print(f"  {LAYER_NAMES[li]}: {np.mean(rv_concordance[li]):.3f}  "
          f"(AUC={np.mean(rv_auc[li]):.3f})")
print()
print("Interpretation:")
print("  Concordance >> 0.5: reading vector generalises across topics")
print("  => model encodes harm content beyond topic")
print("  Concordance ~ 0.5: only topic-level signal, no harm-specific encoding")